In [ ]:
"""
Визуализация результатов детекции
"""

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO
import random

OUTPUT_DIR = Path("../outputs/results/visualizations")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ ДЕТЕКЦИИ")
print("=" * 60)

# Загружаем лучшую модель
model = YOLO("../models/best_yolo8n.pt")

# Пути к тестовым данным
test_images_dir = Path("../data/data/test/images")
test_images = list(test_images_dir.glob("*.jpg")) + list(test_images_dir.glob("*.png"))

# Выбираем 12 случайных изображений
selected_images = random.sample(test_images, min(12, len(test_images)))

fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for idx, img_path in enumerate(selected_images):
    row = idx // 4
    col = idx % 4
    
    # Загружаем изображение
    image = cv2.imread(str(img_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Делаем предсказание
    results = model(image, verbose=False)
    
    # Рисуем результат
    annotated = results[0].plot()
    
    axes[row, col].imshow(annotated)
    axes[row, col].set_title(f"{img_path.name[:20]}")
    axes[row, col].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'detection_results.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Визуализация сохранена: {OUTPUT_DIR / 'detection_results.png'}")

# Создаем коллаж с разными уровнями уверенности
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

confidence_levels = [
    (0.9, "Высокая уверенность (>0.9)"),
    (0.7, "Средняя уверенность (0.7-0.9)"),
    (0.5, "Низкая уверенность (0.5-0.7)")
]

for idx, (conf_threshold, title) in enumerate(confidence_levels):
    # Находим изображения с разной уверенностью
    for img_path in random.sample(test_images, min(10, len(test_images))):
        image = cv2.imread(str(img_path))
        results = model(image, verbose=False)
        
        max_conf = 0
        for r in results:
            if r.boxes is not None:
                for box in r.boxes:
                    conf = float(box.conf[0])
                    max_conf = max(max_conf, conf)
        
        if conf_threshold - 0.1 <= max_conf <= conf_threshold + 0.1:
            # Отображаем
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            annotated = results[0].plot()
            axes[0, idx].imshow(annotated)
            axes[0, idx].set_title(title)
            axes[0, idx].axis('off')
            break

# Пустые изображения для второго ряда
for idx in range(3):
    axes[1, idx].axis('off')
    axes[1, idx].set_title("Примеры ошибок")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confidence_levels.png', dpi=150, bbox_inches='tight')
plt.show()